In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import STL

import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

#  Модель ARMA

До 1980-х годов процессы ARIMA были рабочими лошадками анализа временных рядов, что для многих означало просто поиск правильного порядка такой модели. Этот класс процессов определяется посредством линейной зависимости между факторами наблюдения и шума.

Линейный процесс - это временной ряд $y_t$, определяемый следующим образом:
\begin{equation}
   y_t = \sum_{u = -\infty}^{\infty} \psi_u \epsilon_{t-u}
\end{equation}
где $\epsilon_t$ — ряд белого шума и
\begin{equation}
\sum_{u = -\infty}^{\infty} \left|\psi_u \right|^2 < \infty
\end{equation}


## AR процесс

Простейшим (нетривиальным) примером линейного процесса является авторегрессионный процесс порядка $p$:

\begin{equation}
    y_t = \sum_{i= 1}^p \phi_i y_{t-i} + \epsilon_t
\end{equation}
    
Пояснение:
1. Естественное расширение модели множественной линейной регрессии: мы прогнозируем интересующую переменную, используя линейную комбинацию прошлых значений переменной (лагированные значения действуют как предикторы). Термин \textit{auto}регрессия отражает тот факт, что мы регрессируем переменную к самой себе (версии).
2. Легко оценить параметры и сделать прогноз.


Теперь, когда мы определили, что такое процесс AR, давайте посмотрим, как его идентифицировать. Это актуально, если нам нужно решить, является ли процесс авторегрессии подходящей моделью для конкретного набора данных. Для демонстрации мы будем использовать смоделированные данные, где процесс генерации данных:

\begin{equation}
    y_t = 0.9 y_{t-1} + \epsilon_t
\end{equation}
    
Обратите внимание, что в кодовом блоке ниже коэффициент меняет знак — это связано с соглашением об обозначениях в статистических моделях, где мы читаем коэффициенты в порядке, определенном характеристическим полиномом (подробнее об этом ниже):

\begin{equation}
    y_t - 0.9 y_{t-1} = \epsilon_t
\end{equation}

Быстрый способ определить, является ли AR подходящей моделью для данного ряда, — это изучить функции автокорреляции и частичной автокорреляции

In [ ]:
ar1 = np.array([1, -0.9])
ma1 = np.array([1])
AR_object1 = ArmaProcess(ar1, ma1)
simulated_data_1 = AR_object1.generate_sample(nsample=300)
plt.figure(figsize=(16,5), dpi=300)
plt.plot(simulated_data_1)
plt.show()

In [ ]:
plt.figure(figsize=(10,7), dpi=100)
ax = plt.subplot(211)
sm.graphics.tsa.plot_acf(simulated_data_1, lags=10, ax=ax)
ax = plt.subplot(212)
sm.graphics.tsa.plot_pacf(simulated_data_1, lags=10, ax=ax)
plt.show()

У процесса AR(p):
   - ACF экспоненциально убывает,
   - PACF обрывается на лаге $p$.


Проверим процесс на стационарность: если корни процесса авторегрессии находятся за пределами единичного круга, то `True`. 

In [ ]:
AR_object1.isstationary

Посмотрим на корни характеристического полинома оператора запаздывания для авторегрессии

In [ ]:
AR_object1.arroots

## MA процесс

Модель авторегрессии выражает переменную прогноза как линейную комбинацию прошлых реализаций самой себя, и та же идея может быть применена к прошлым ошибкам прогноза: процесс скользящего среднего порядка $q$ определяется соотношением:
\begin{equation}
y_t = \epsilon_t + \sum_{i = 1}^q \theta_i \epsilon_{t-i}
\end{equation}
где $\epsilon_t$ — ряд белого шума.

Давайте повторим предыдущее упражнение по идентификации:
\begin{equation}
    y_t = \epsilon_t + 0.9  \epsilon_{t-1}
\end{equation}

In [ ]:
ar1 = np.array([1])
ma1 = np.array([1, -0.9])
MA_object1 = ArmaProcess(ar1, ma1)
simulated_data_2 = MA_object1.generate_sample(nsample=300)
plt.figure(figsize=(16,5), dpi=300)
plt.plot(simulated_data_2)
plt.show()

In [ ]:
plt.figure(figsize=(10,7), dpi=100)
ax = plt.subplot(211)
sm.graphics.tsa.plot_acf(simulated_data_2, lags=10, ax=ax)
ax = plt.subplot(212)
sm.graphics.tsa.plot_pacf(simulated_data_2, lags=10, ax=ax)
plt.show()

У процесса MA(q):
   - ACF обрывается на лаге $q$,
   - PACF экспоненциально убывает.

Проверим процесс на обратимость: если корни процесса скользящего среднего находятся за пределами единичного круга, то `True`.

In [ ]:
MA_object1.isinvertible

In [ ]:
MA_object1.maroots

## ARMA процесс

У нас есть авторегрессионный компонент и компонент скользящего среднего, поэтому вполне естественно объединить эти два типа динамики в одну модель: ряд ARMA(p,q) удовлетворяет соотношению:
\begin{equation}
y_t = \sum_{i=1}^p \phi_i y_{t-i} + \sum_{j=1}^q \theta_j \epsilon_{t-j} + \epsilon_t,
\end{equation}

где наши предикторы в правой части включают как запаздывающие значения ряда, так и запаздывающие ошибки, $p$ — это порядок авторегрессионной части, а $q$ — это порядок компонента скользящего среднего.

1. если модель ARMA(p,q) является стационарной, ее можно представлен как бесконечная серия AR
\begin{equation}    
    y_t = \sum_{u=1}^\infty \pi_u y_{t-u} + \epsilon_t
\end{equation}

2. возможно оценивать парамерты с помощью метода максимального правдоподобия и простого рекурсивного прогноза:
\begin{equation}
    \hat{y}_{T+1} = \sum_{u=1}^\infty \hat{\pi}_u y_{T+1 - u}
\end{equation}
3. Стационарность процесса ARMA(p,q) устанавливается путем анализа характеристического многочлена. 
\begin{equation}
P(z) = 1 - \phi_1 z - \ldots - \phi_p z^p
\end{equation}
Ищем решения в комплексной области: уравнение $P(z) = 0$ имеет $p$ решений $z_1, \ldots, z_p$
       
     * если $|z_i| >1$ для всех $i$, то модель стационарна
     * критерии, такие как критерий Дики-Фуллера, проверяют наличие единичных корней

Работать будем с данными о количество преступлений в России

In [ ]:
crime = pd.read_csv('crime.csv', parse_dates=['month'], dayfirst=True)
crime.set_index('month', inplace = True)

In [ ]:
plt.figure(figsize=(16,5), dpi=300)
plt.plot(crime.index, crime.Total_crimes, color='tab:blue')
plt.gca().set(title='Количество преступлений в России', xlabel='Date', ylabel='Value')
plt.show()

Воспольуземся критерием Дики-Фуллера для проверки стационарности ряда:

**Тест Дики-Фуллера**:

 $H_0\colon$ ряд не стационарен
 
 $H_1\colon$ ряд стационарен


Напомним, как по $p-value$ определить, есть ли основания отвергнуть нулевую гипотезу. 

1. Зафиксируем уровень значимости $\alpha$ – это вероятность отвергнуть нулевую гипотезу при условии, что она верна.
2. $p-value$ – это минимальный уровень значимости, на котором нулевая гипотеза может быть отвергнута.

Значит, если:

$p−value < \alpha$ $\Rightarrow$ $H_0$ отвергаем на уровне значимости $\alpha$ (на имеющихся данных)

$p−value \geq \alpha$ $\Rightarrow$ $H_0$ не отвергаем на уровне значимости $\alpha$ (на имеющихся данных)

In [ ]:
print('Критерий Дики-Фуллера =', round(sm.tsa.stattools.adfuller(crime.Total_crimes)[1], 4))

In [ ]:
stl = STL(crime.Total_crimes, seasonal=13, robust=True) 
result_stl = stl.fit()

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16,5))
 
ax1.plot(crime.index, result_stl.trend, label='Trend', color='red')
ax1.set_title('Тренд')
ax2.plot(crime.index, result_stl.seasonal, label='Seasonal', color='blue')
ax2.set_title('Сезонность')
ax3.plot(crime.index, result_stl.resid, label='Residual')
ax3.set_title('Случайные колебания')
plt.tight_layout()
plt.show()

In [ ]:
detrended_series = crime.Total_crimes - result_stl.trend
 
plt.figure(figsize=(16,5))
plt.plot(crime.index, detrended_series, label='Ряд после удаления тренда', color='green')
plt.xlabel('Year')
plt.ylabel('Количество преступлений')
plt.legend()
plt.show()

In [ ]:
print('Критерий Дики-Фуллера =', round(sm.tsa.stattools.adfuller(detrended_series)[1], 4))

In [ ]:
plt.figure(figsize=(10,7), dpi=100)
ax = plt.subplot(211)
sm.graphics.tsa.plot_acf(detrended_series, lags=36, ax=ax)
ax = plt.subplot(212)
sm.graphics.tsa.plot_pacf(detrended_series, lags=36, ax=ax)
plt.show()

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
model = SARIMAX(detrended_series, 
                order = (1, 0, 1))

In [ ]:
result = model.fit()

In [ ]:
result.summary()

In [ ]:
start = len(detrended_series)
end = len(detrended_series) + 12 - 1

In [ ]:
predictions = result.predict(start, end)

In [ ]:
predictions

In [ ]:
plt.figure(figsize=(16,5))

plt.plot(detrended_series, color = "green")
plt.plot(predictions, color = "black")
 
# заголовок и подписи к осям
plt.title("Данные и прогноз")
plt.ylabel('Количество преступлений')
plt.xlabel('Год')
 
# добавим сетку
plt.grid()
 
plt.show()